# Advanced Analytics for Mutual Funds

This notebook computes Value-at-Risk, Conditional VaR, rolling Sharpe, investor cohorts, SIP continuity risk, sector concentration, and fund recommendations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
BASE = Path('.')
PROCESSED = BASE / 'data' / 'processed'
nav = pd.read_csv(PROCESSED / '02_nav_history.csv', parse_dates=['date']).sort_values(['amfi_code','date'])
perf = pd.read_csv(PROCESSED / '07_scheme_performance.csv')
transactions = pd.read_csv(PROCESSED / '08_investor_transactions.csv', parse_dates=['transaction_date']).sort_values(['investor_id','transaction_date'])
holdings = pd.read_csv(PROCESSED / '09_portfolio_holdings.csv')
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
var_cvar = pd.read_csv('var_cvar_report.csv')
rolling = pd.read_csv('rolling_sharpe_data.csv') if (BASE / 'rolling_sharpe_data.csv').exists() else None
print('Data loaded. Unique funds:', nav['amfi_code'].nunique())

## Historical VaR and CVaR

The 95% VaR is the 5th percentile of the daily return distribution per fund. CVaR is the mean return for outcomes below the VaR threshold.

In [ ]:
var_cvar = pd.read_csv('var_cvar_report.csv')
var_cvar[['scheme_name','var_95_pct','cvar_95_pct','observations']].sort_values('var_95_pct').head(10)

## Rolling 90-Day Sharpe Ratio

This chart tracks the rolling Sharpe for 5 key funds over time.

In [ ]:
from PIL import Image
img = Image.open('rolling_sharpe_chart.png')
display(img)

## Investor Cohort Analysis

Cohorts are defined by the year of each investor's first SIP transaction. We compute average SIP amount, total invested, and each cohort's top fund preference.

In [ ]:
cohort = pd.read_csv('investor_cohort_report.csv')
cohort.head(10)

## SIP Continuity Analysis

Investors with 6+ SIPs are evaluated on average gap between SIP dates. Investors with average gap > 35 days are flagged as at-risk.

In [ ]:
continuity = pd.read_csv('sip_continuity_report.csv')
continuity.head(10)

## Sector Concentration (HHI)

Herfindahl-Hirschman Index measures portfolio concentration. Higher HHI implies a more concentrated equity portfolio.

In [ ]:
hhi = pd.read_csv('hhi_concentration_report.csv')
hhi.head(10)

## Fund Recommender

A simple risk-appetite-based recommender selects the top 3 funds by Sharpe ratio within the chosen risk grade.

In [ ]:
import subprocess, sys
print('Run recommender.py with a risk appetite: Low, Moderate, or High')

## Advanced Insights

1. The funds with the lowest 95% VaR are typically the most stable large-cap funds, while the highest CVaR appears in more concentrated aggressive schemes.
2. Investor cohorts formed in later years show higher average SIP amounts, indicating stronger systematic investment behavior among newer retail cohorts.
3. SIP continuity risk is concentrated among investors with long SIP histories and average gaps above 35 days, suggesting attention is needed for retention and autopay reminders.
4. Equity funds with the highest HHI are the most concentrated portfolios; these funds may offer strong conviction but higher single-stock exposure.
5. Risk-grade based recommendations favor high Sharpe ratio funds within each appetite bucket, making it easy to compare conservative, moderate, and aggressive fund choices.